# 05 - Finalize, validate & package the Caravan bundle for Zenodo

Run **after `04_caravan_part2_postprocess`**. End to end it makes `04_caravan_bundle` match the
[Caravan "Sharing New Data"](https://github.com/kratzert/Caravan/wiki/Sharing-New-Data) layout and produces the upload zip:

1. Refresh `attributes_additional_ausvic.csv` from the bundle's own timeseries (periods always match the data).
2. Write `licenses/ausvic/license_ausvic.md` (from `notebooks/05_assets/`) + copy the project-root `README.md` and `outlet_proof.png` into the bundle.
3. Copy the simplified shapefile into the bundle.
4. Validate the full structure (4 required folders, each holding only `ausvic/`; 6-col `attributes_other`; gauge_id consistency; no negative/>150 streamflow; shapefile gauge_id-only EPSG:4326) with a hard assert.
5. Zip into `05_zenodo/caravan_ausvic.zip` - ready to upload.

Idempotent; safe to re-run.

In [1]:
from pathlib import Path
import pandas as pd, geopandas as gpd, glob, os, shutil, zipfile

ROOT          = Path('C:/Users/leela/FloodHubMaribyrnong')
BUNDLE_DIR    = ROOT / '04_caravan_bundle'
SHAPEFILE_DIR = ROOT / '01_shapefile' / 'outputs'
ASSETS_DIR    = ROOT / 'notebooks' / '05_assets'        # editable license + README sources
ZENODO_DIR    = ROOT / '05_zenodo'                      # the upload zip lands here
PREFIX        = 'ausvic'
ZIP_NAME      = 'caravan_ausvic'                        # zip filename + root folder inside it

ATTR_DIR = BUNDLE_DIR / 'attributes' / PREFIX
TS_DIR   = BUNDLE_DIR / 'timeseries' / 'csv' / PREFIX
print('bundle :', BUNDLE_DIR)
print('zip out:', ZENODO_DIR / f'{ZIP_NAME}.zip')

bundle : C:\Users\leela\FloodHubMaribyrnong\04_caravan_bundle
zip out: C:\Users\leela\FloodHubMaribyrnong\05_zenodo\caravan_ausvic.zip


## 1 - Refresh `attributes_additional` from the bundle timeseries

In [2]:
GAUGE_META = {
    'ausvic_230119': ('Deep Creek',        'bom',     ''),
    'ausvic_230100': ('Deep Creek',        'bom',     ''),
    'ausvic_230107': ('Deep Creek',        'bom',     'Mixed BoM quality (~51% quality-A, ~34% quality-E).'),
    'ausvic_230102': ('Deep Creek',        'bom',     ''),
    'ausvic_230237': ('Maribyrnong River', 'bom',     'Oct-2022 flood: the daily-mean peak here (~313 m3/s) is low against both upstream (Bulla + Sunbury, ~364 m3/s combined) and downstream (Keilor, ~503 m3/s, corroborated by the Jacobs 2023 post-event analysis); likely high-stage rating underestimate. Treat major-flood peaks at this gauge with caution.'),
    'ausvic_230106': ('Maribyrnong River', 'bom',     'Tidal site: BoM certifies discharge in the flood range (quality-A, incl. Oct-2022 peak); tidal baseflow recorded as uncertified ~0. Reliable flood-event record, low-confidence baseline.'),
    'ausvic_230211': ('Bolinda Creek',     'bom',     ''),
    'ausvic_230200': ('Maribyrnong River', 'hydstra', 'Record starts 1908-02-02; rows before 1950-01-02 carry streamflow only, with empty ERA5-Land forcing columns (Caravan convention for pre-reanalysis records).'),
    'ausvic_230206': ('Jacksons Creek',    'hydstra', 'Regulated: downstream of Rosslynne Reservoir, the catchment\'s only major storage (built 1971-74, upper Jacksons Creek). Major events can be wholly absorbed while the storage fills: the May-1974 flood (third-largest at Keilor) is absent at this gauge, and runoff is strongly suppressed during post-drought refill years (e.g. 2000s). Treat flood peaks here as regulated, not natural.'),
    'ausvic_230202': ('Jacksons Creek',    'hydstra', 'Hydstra record from 1908 but pre-1960 are level-derived artefacts (no rating); kept from 1960.'),
}
SRC = {'bom':     'Bureau of Meteorology, Water Data Online (Water Course Discharge, DMQaQc.Merged; sub-daily -> daily mean)',
       'hydstra': 'Victorian Government Water Measurement Information System (data.water.vic.gov.au; daily mean discharge, var 141.00)'}
DEF_NOTE = {'bom':     'BoM sub-daily discharge aggregated to a time-weighted daily mean; near-zero negative sensor values clipped to 0.',
            'hydstra': 'Victorian Water daily mean discharge (var 141.00); near-zero negatives clipped to 0.'}

rows = []
for gid, (basin, prov, note) in GAUGE_META.items():
    s = pd.to_numeric(pd.read_csv(TS_DIR / f'{gid}.csv', parse_dates=['date']).set_index('date')['streamflow'], errors='coerce')
    fv, lv = s.first_valid_index(), s.last_valid_index()
    period_days = (lv - fv).days + 1
    valid = int(s.loc[fv:lv].notna().sum())
    rows.append({'gauge_id': gid, 'basin_name': basin, 'unit_area': 'km2',
                 'streamflow_period': f'{fv.date()}/{lv.date()}', 'streamflow_missing': round(1 - valid / period_days, 4),
                 'streamflow_units': 'mm/d', 'source': SRC[prov], 'license': 'CC-BY-4.0',
                 'note': note or DEF_NOTE[prov]})
add = pd.DataFrame(rows).set_index('gauge_id').sort_index()
# Plain UTF-8 -- never 'utf-8-sig': a BOM makes the header read back as '\ufeffgauge_id'
# in a default pd.read_csv, which silently breaks gauge_id joins downstream.
add.to_csv(ATTR_DIR / f'attributes_additional_{PREFIX}.csv')
print('attributes_additional refreshed ->', add.shape)
print(add[['basin_name', 'streamflow_period', 'streamflow_missing']].to_string())

attributes_additional refreshed -> (10, 8)
                      basin_name      streamflow_period  streamflow_missing
gauge_id                                                                   
ausvic_230100         Deep Creek  1975-09-04/2026-03-06              0.0643
ausvic_230102         Deep Creek  1955-06-21/2026-03-06              0.0170
ausvic_230106  Maribyrnong River  1975-09-03/2026-03-06              0.0215
ausvic_230107         Deep Creek  1979-10-11/2026-03-06              0.1680
ausvic_230119         Deep Creek  2000-02-09/2026-03-06              0.0858
ausvic_230200  Maribyrnong River  1908-02-02/2026-03-06              0.1538
ausvic_230202     Jacksons Creek  1960-01-01/2026-03-06              0.0000
ausvic_230206     Jacksons Creek  1960-05-10/2026-03-06              0.0000
ausvic_230211      Bolinda Creek  1975-06-04/2026-03-06              0.1121
ausvic_230237  Maribyrnong River  1998-12-09/2026-03-06              0.0213


## 2 - Write `licenses/ausvic/license_ausvic.md` + dated `README.md`

In [3]:
# 2a. license -> licenses/ausvic/license_ausvic.md  (verbatim from the editable source)
lic_dir = BUNDLE_DIR / 'licenses' / PREFIX
lic_dir.mkdir(parents=True, exist_ok=True)
shutil.copy(ASSETS_DIR / f'license_{PREFIX}.md', lic_dir / f'license_{PREFIX}.md')

# 2b. README.md  (single canonical README lives at the project root; copy it into the bundle)
dmin = dmax = None
for p in glob.glob(str(TS_DIR / '*.csv')):
    d = pd.to_datetime(pd.read_csv(p, usecols=['date'])['date'])
    dmin = d.min() if dmin is None else min(dmin, d.min())
    dmax = d.max() if dmax is None else max(dmax, d.max())
shutil.copy(ROOT / 'README.md', BUNDLE_DIR / 'README.md')
shutil.copy(SHAPEFILE_DIR / 'outlet_proof.png', ROOT / 'outlet_proof.png')        # refresh repo-root copy
shutil.copy(SHAPEFILE_DIR / 'outlet_proof.png', BUNDLE_DIR / 'outlet_proof.png')  # bundle copy for the README image

# 2c. root LICENSE.txt: a one-page pointer (the full attributed license stays at
#     licenses/ausvic/license_ausvic.md, matching the Caravan licenses/ layout).
(BUNDLE_DIR / 'LICENSE.txt').write_text(
    'Caravan extension "ausvic" -- Maribyrnong River basin, Victoria, Australia\n'
    'License: Creative Commons Attribution 4.0 International (CC-BY-4.0)\n'
    'https://creativecommons.org/licenses/by/4.0/\n'
    '\n'
    'Per-source attributions, references and the citation line:\n'
    '  licenses/ausvic/license_ausvic.md\n',
    encoding='utf-8')
shutil.rmtree(BUNDLE_DIR / 'temp', ignore_errors=True)
print('license ->', lic_dir / f'license_{PREFIX}.md')
print('README period:', dmin.date(), '->', dmax.date())

license -> C:\Users\leela\FloodHubMaribyrnong\04_caravan_bundle\licenses\ausvic\license_ausvic.md
README period: 1908-02-02 -> 2026-03-06


## 3 - Copy the simplified shapefile into the bundle

In [4]:
sd = BUNDLE_DIR / 'shapefiles' / PREFIX
sd.mkdir(parents=True, exist_ok=True)
for ext in ['shp', 'shx', 'dbf', 'prj', 'cpg']:
    src = SHAPEFILE_DIR / f'{PREFIX}_basin_shapes.{ext}'
    if src.is_file():
        shutil.copy(src, sd / src.name)
print('shapefile siblings in bundle:', sorted(p.name for p in sd.glob(f'{PREFIX}_basin_shapes.*')))

shapefile siblings in bundle: ['ausvic_basin_shapes.cpg', 'ausvic_basin_shapes.dbf', 'ausvic_basin_shapes.prj', 'ausvic_basin_shapes.shp', 'ausvic_basin_shapes.shx']


## 4 - Validate the bundle against the Caravan layout

In [5]:
import numpy as np
from shapely.geometry import Point

def ids(p): return set(pd.read_csv(p)['gauge_id'])
def names(d): return sorted(x.name for x in d.iterdir() if not x.name.startswith('.'))
shp_path = BUNDLE_DIR / 'shapefiles' / PREFIX / f'{PREFIX}_basin_shapes.shp'
checks = {}

# ---- files & layout ---------------------------------------------------------
for f in ['attributes_other', 'attributes_hydroatlas', 'attributes_caravan', 'attributes_additional']:
    checks[f'file: {f}'] = (ATTR_DIR / f'{f}_{PREFIX}.csv').is_file()
checks['file: shapefile'] = shp_path.is_file()
checks['file: license_ausvic.md'] = (BUNDLE_DIR / 'licenses' / PREFIX / f'license_{PREFIX}.md').is_file()
checks['file: LICENSE.txt (root)'] = (BUNDLE_DIR / 'LICENSE.txt').is_file()
checks['file: outlet_proof.png'] = (BUNDLE_DIR / 'outlet_proof.png').is_file()
checks['attributes/ = [ausvic]'] = names(BUNDLE_DIR / 'attributes') == [PREFIX]
checks['shapefiles/ = [ausvic]'] = names(BUNDLE_DIR / 'shapefiles') == [PREFIX]
checks['licenses/ = [ausvic]']   = names(BUNDLE_DIR / 'licenses') == [PREFIX]
checks['timeseries/ = [csv, netcdf]']   = names(BUNDLE_DIR / 'timeseries') == ['csv', 'netcdf']
checks['timeseries/csv/ = [ausvic]']    = names(BUNDLE_DIR / 'timeseries' / 'csv') == [PREFIX]
checks['timeseries/netcdf/ = [ausvic]'] = names(BUNDLE_DIR / 'timeseries' / 'netcdf') == [PREFIX]

# ---- encoding: no UTF-8 BOM anywhere (a BOM breaks default pd.read_csv keying) ----
bom_files = []
for p in glob.glob(str(ATTR_DIR / '*.csv')) + glob.glob(str(TS_DIR / '*.csv')):
    with open(p, 'rb') as fh:
        if fh.read(3) == b'\xef\xbb\xbf':
            bom_files.append(os.path.basename(p))
checks['no UTF-8 BOM in any csv'] = bom_files == []
if bom_files:
    print('  BOM found in:', ', '.join(bom_files))

# ---- attributes_other: exact Caravan columns IN ORDER, ISO alpha-2 country ----
ao = pd.read_csv(ATTR_DIR / f'attributes_other_{PREFIX}.csv')
checks['attributes_other order = gauge_id,gauge_name,country,gauge_lat,gauge_lon,area'] = \
    list(ao.columns) == ['gauge_id', 'gauge_name', 'country', 'gauge_lat', 'gauge_lon', 'area']
checks["country = 'AU' everywhere"] = 'country' in ao.columns and bool((ao['country'] == 'AU').all())
checks['gauge_id = provider_id, one underscore'] = all(
    g.count('_') == 1 and g.split('_')[0] == PREFIX for g in ao['gauge_id'])

base = set(ao['gauge_id'])
for f in ['attributes_hydroatlas', 'attributes_caravan', 'attributes_additional']:
    checks[f'gauge_id match: {f}'] = ids(ATTR_DIR / f'{f}_{PREFIX}.csv') == base
ts = set(os.path.basename(p)[:-4] for p in glob.glob(str(TS_DIR / '*.csv')))
nc = set(os.path.basename(p)[:-3] for p in glob.glob(str(BUNDLE_DIR / 'timeseries' / 'netcdf' / PREFIX / '*.nc')))
g = gpd.read_file(shp_path)
checks['gauge_id match: timeseries/csv']    = ts == base
checks['gauge_id match: timeseries/netcdf'] = nc == base
checks['gauge_id match: shapefile']         = set(g['gauge_id']) == base

# ---- shapefile: CRS, columns, UNIQUE polygons, areas, gauges at outlets ----
checks['shapefile gauge_id-only'] = list(g.columns) == ['gauge_id', 'geometry']
checks['shapefile EPSG:4326'] = (g.crs is not None and g.crs.to_epsg() == 4326)
checks['every polygon unique'] = g.geometry.apply(lambda x: x.wkb).nunique() == len(g)
gm = g.merge(ao, on='gauge_id').to_crs(epsg=3577)        # GDA94 Australian Albers, equal-area
poly_km2 = gm.geometry.area / 1e6
checks['polygon area = attributes_other.area (<1%)'] = bool(
    ((poly_km2 - gm['area']).abs() / gm['area'] < 0.01).all())
pts = gpd.GeoSeries([Point(xy) for xy in zip(gm['gauge_lon'], gm['gauge_lat'])], crs=4326).to_crs(3577)
d_edge = pts.distance(gm.geometry.boundary)
d_cent = pts.distance(gm.geometry.centroid)
checks['every gauge at its polygon OUTLET (<=1 km from edge, edge << centroid)'] = bool(
    (d_edge <= 1000).all() and (d_edge < 0.25 * d_cent).all())

# ---- timeseries content -----------------------------------------------------
neg = hi = 0
degenerate_last, forcing_gap, bad_pre = [], [], []
for p in sorted(glob.glob(str(TS_DIR / '*.csv'))):
    gid = os.path.basename(p)[:-4]
    df = pd.read_csv(p, parse_dates=['date'])
    v = pd.to_numeric(df['streamflow'], errors='coerce')
    neg += int((v < 0).sum()); hi += int((v > 150).sum())
    last = df.iloc[-1]
    if (last['temperature_2m_min'] == last['temperature_2m_max']
            and last['surface_pressure_min'] == last['surface_pressure_max']):
        degenerate_last.append(gid)
    era5 = df.loc[df['date'] >= '1950-01-02']
    if era5['total_precipitation_sum'].isna().any() or era5['temperature_2m_mean'].isna().any():
        forcing_gap.append(gid)
    pre = df.loc[df['date'] < '1950-01-02']
    if len(pre) and pre['temperature_2m_mean'].notna().any():
        bad_pre.append(gid)            # pre-ERA5 rows must carry streamflow ONLY
checks['no negative streamflow'] = neg == 0
checks['no >150 mm/d streamflow'] = hi == 0
checks['no degenerate trailing ERA5 day (partial-day trim worked)'] = degenerate_last == []
checks['forcings complete over the full ERA5 period'] = forcing_gap == []
checks['pre-1950 rows (if any) have EMPTY forcings'] = bad_pre == []

# Keilor must include its pre-ERA5 record again (1908+), per the Caravan convention.
k = pd.read_csv(TS_DIR / 'ausvic_230200.csv', parse_dates=['date'])
kpre = pd.to_numeric(k.loc[k['date'] < '1950-01-02', 'streamflow'], errors='coerce')
checks['ausvic_230200 pre-1950 streamflow restored (starts 1908, >8000 valid days)'] = \
    len(k) and k['date'].min() < pd.Timestamp('1910-01-01') and int(kpre.notna().sum()) > 8000

# ---- hydroatlas: official-notebook output -----------------------------------
ha_cols = pd.read_csv(ATTR_DIR / f'attributes_hydroatlas_{PREFIX}.csv', nrows=0).columns
checks['hydroatlas: gauge_id first + ~196 BasinATLAS attrs'] = \
    ha_cols[0] == 'gauge_id' and 190 <= len(ha_cols) - 1 <= 200

print('BUNDLE VALIDATION'); print('=' * 60)
for kk, vv in checks.items():
    print(f'  {"PASS" if vv else "FAIL"}  {kk}')
print(f'\n  note: hydroatlas attribute count = {len(ha_cols) - 1} '
      f'(state this in the submission reply; GEE BasinATLAS gains properties over time)')
bad = [kk for kk, vv in checks.items() if not vv]
assert not bad, 'VALIDATION FAILED: ' + ', '.join(bad)
print(f'\nAll {len(checks)} checks passed -- bundle is submission-ready.')

BUNDLE VALIDATION
  PASS  file: attributes_other
  PASS  file: attributes_hydroatlas
  PASS  file: attributes_caravan
  PASS  file: attributes_additional
  PASS  file: shapefile
  PASS  file: license_ausvic.md
  PASS  file: LICENSE.txt (root)
  PASS  file: outlet_proof.png
  PASS  attributes/ = [ausvic]
  PASS  shapefiles/ = [ausvic]
  PASS  licenses/ = [ausvic]
  PASS  timeseries/ = [csv, netcdf]
  PASS  timeseries/csv/ = [ausvic]
  PASS  timeseries/netcdf/ = [ausvic]
  PASS  no UTF-8 BOM in any csv
  PASS  attributes_other order = gauge_id,gauge_name,country,gauge_lat,gauge_lon,area
  PASS  country = 'AU' everywhere
  PASS  gauge_id = provider_id, one underscore
  PASS  gauge_id match: attributes_hydroatlas
  PASS  gauge_id match: attributes_caravan
  PASS  gauge_id match: attributes_additional
  PASS  gauge_id match: timeseries/csv
  PASS  gauge_id match: timeseries/netcdf
  PASS  gauge_id match: shapefile
  PASS  shapefile gauge_id-only
  PASS  shapefile EPSG:4326
  PASS  every pol

## 5 - Package the Zenodo upload zip

In [6]:
ZENODO_DIR.mkdir(parents=True, exist_ok=True)
zip_path = ZENODO_DIR / f'{ZIP_NAME}.zip'
if zip_path.exists():
    zip_path.unlink()

include_dirs  = ['attributes', 'timeseries', 'shapefiles', 'licenses']   # the 4 required Caravan folders
include_files = ['README.md', 'outlet_proof.png', 'LICENSE.txt']
n = 0
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in include_files:
        if (BUNDLE_DIR / f).is_file():
            z.write(BUNDLE_DIR / f, f'{ZIP_NAME}/{f}'); n += 1
    for d in include_dirs:
        for p in sorted((BUNDLE_DIR / d).rglob('*')):
            if p.is_file():
                z.write(p, f'{ZIP_NAME}/' + p.relative_to(BUNDLE_DIR).as_posix()); n += 1

mb = zip_path.stat().st_size / 1e6
with zipfile.ZipFile(zip_path) as z:
    top = sorted({e.split('/')[1] for e in z.namelist() if e.count('/') >= 1})
print(f'wrote {zip_path}')
print(f'  {n} files, {mb:.1f} MB')
print(f'  zip root: {ZIP_NAME}/')
print(f'  contains: {", ".join(top)}')
print('\nReady to upload to Zenodo.')

wrote C:\Users\leela\FloodHubMaribyrnong\05_zenodo\caravan_ausvic.zip
  33 files, 33.7 MB
  zip root: caravan_ausvic/
  contains: LICENSE.txt, README.md, attributes, licenses, outlet_proof.png, shapefiles, timeseries

Ready to upload to Zenodo.
